# 4.02 Eigenmode and EPR analysis
## Prerequisite
You need to have a working local installation of Ansys.

## Sections
### I. Transmon only
1. Prepare the layout in qiskit-metal. <br>
1. Run finite element eigenmode analysis. <br>
1. Plot fields and display them. <br>
1. Set up EPR junction dictionary. <br>
1. Run EPR analysis on single mode. <br>
1. Get qubit freq and anharmonicity. <br>
1. Calculate EPR of substrate.  <br>
1. (Extra: Calculate surface EPR.) <br>

### II. Resonator only
1. Update the layout in qiskit-metal. <br>
1. Run finite element eigenmode analysis. <br>
1. Plot fields and display them. <br>
1. Calculate EPR of substrate. <br>
    
### III. Transmon & resonator
1. Update the layout in qiskit-metal. <br>
1. Run finite element eigenmode analysis. <br>
1. Plot fields and display them. <br>
1. Set up EPR junction dictionary. <br>
1. Run EPR analysis on the two modes. <br>
1. Get qubit frequency and anharmonicity. <br>

### IV. Analyze a coupled 2 transmon system. 
1. Finite Element Eigenmode Analysis
1. Identify the mode you want. The mode can inclusively be from 1 to setup.n_modes.
1. Set variables in the Ansys design. As before, we seek 2 modes.
1. Set up the simulation and specify the variables for the sweep.
1. Plot the E-field on the chip's surface.
1. Specify the junctions in the model; in this case there are 2 junctions.
1. Find the electric and magnetic energy stored in the substrate and the system as a whole.
1. Perform EPR analysis for all modes and variations.

## 1. Analyze the transmon qubit by itself

We will use the analysis package - applicable to most users. Advanced users might want to expand the package, or directly interact with the renderer. The renderer is one of the properties of the analysis class.

### Create the Qbit design

Setup a design of a given dimension. Dimensions will be respected in the design rendering. <br>
Note that the design size extends from the origin into the first quadrant.

In [16]:
import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, Headings
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
design = designs.DesignPlanar({}, True)
design.chips.main.size["size_x"] = "7mm"
design.chips.main.size["size_y"] = "3mm"

gui = MetalGUI(design)

Create a single transmon with one readout resonator and move it to the center of the chip previously defined.

In [17]:
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven


design.delete_all_components()

q1 = TransmonPocket(
    design,
    "Q1",
    options=dict(
        pos_x = '2mm',
        pad_width="365 um",
        pocket_height="650um",
        connection_pads=dict(
            readout=dict(loc_W=-1, loc_H=+1, pad_width="100um"),
            a=dict(loc_W=+1, loc_H=+1, pad_width="100um"),
        ),
    ),
)

q2 = TransmonPocket(
    design,
    "Q2",
    options=dict(
        pos_x='5mm',
        pad_width="367 um",
        pocket_height="650um",
        connection_pads=dict(
            a=dict(loc_W=-1, loc_H=+1, pad_width="180um"),
            readout2=dict(loc_W=+1, loc_H=+1, pad_width="130um"),
        ),
    ),
)

coupler = RouteMeander(
    design,
    "coupler",
    Dict(
        total_length="10.1mm",
        hfss_wire_bonds=True,
        fillet="90um",
        lead=Dict(
            start_straight="0.3mm",
            end_straight="0.3mm",
        ),
        trace_gap="9um",
        trace_width="10um",
        pin_inputs=Dict(
            start_pin=Dict(component="Q1", pin="a"),
            end_pin=Dict(component="Q2", pin="a"),
        ),
    ),
)

otg1 = OpenToGround(
    design,
    "open_to_ground1",
    options=dict(pos_x="0.5mm", pos_y="1.5mm", orientation="0"),
)

otg2 = OpenToGround(
    design,
    "open_to_ground2",
    options=dict(pos_x="6.5mm", pos_y="1.5mm", orientation="180"),
)


port_1 = LaunchpadWirebondDriven(
    design,
    'port_1',
    options=dict(
        pos_x='0',
        pos_y='0.2mm',
        orientation='0',
        pad_width='350um',
        pad_height='129um',
        pad_gap='150um',
        trace_width='10um',
        trace_gap='6um',
        lead_length='9um',
        taper_height='350um',
    ),
)

port_2 = LaunchpadWirebondDriven(
    design,
    'port_2',
    options=dict(
        pos_x='7',
        pos_y='0.2mm',
        orientation='180',
        pad_width='350um',
        pad_height='129um',
        pad_gap='150um',
        trace_width='10um',
        trace_gap='6um',
        lead_length='9um',
        taper_height='350um',
    ),
)

line_1 = RouteStraight(
    design,
    "line_1",
    Dict(
        hfss_wire_bonds=True,
        trace_gap="6um",
        trace_width="10um",
        pin_inputs=Dict(
            start_pin=Dict(component="port_1", pin="tie"),
            end_pin=Dict(component="Q1", pin="readout"),
        ),
    ),
)

line_2 = RouteStraight(
    design,
    "line_2",
    Dict(
        hfss_wire_bonds=True,
        trace_gap="6um",
        trace_width="10um",
        pin_inputs=Dict(
            start_pin=Dict(component="port_2", pin="tie"),
            end_pin=Dict(component="Q2", pin="readout2"),
        ),
    ),
)

gui.rebuild()
gui.autoscale()

### Finite Element Eigenmode Analysis

#### Setup

Select the analysis you intend to run from the `qiskit_metal.analyses` collection.<br>
Select the design to analyze and the tool to use for any external simulation.

In [18]:
from qiskit_metal.analyses.quantization import EPRanalysis

In [19]:
eig_qb = EPRanalysis(design, "hfss")
eig_qb.sim.setup.n_modes = 3

Review and update the convergence parameters and junction properties by executing following two cells. We exemplify three different methods to update the setup parameters.

In [20]:
eig_qb.sim.setup

{'name': 'Setup',
 'reuse_selected_design': True,
 'reuse_setup': True,
 'min_freq_ghz': 1,
 'n_modes': 3,
 'max_delta_f': 0.5,
 'max_passes': 10,
 'min_passes': 1,
 'min_converged': 1,
 'pct_refinement': 30,
 'basis_order': 1,
 'vars': {'Lj': '10 nH', 'Cj': '0 fF'}}

In [21]:
# example: update single setting
eig_qb.sim.setup.max_passes = 6
#eig_qb.sim.setup.vars.Lj = "15.2 nH"
#eig_qb.sim.setup.vars.Cj = "45.7 fF"
eig_qb.sim.setup.vars = Dict(Lj1="13 nH", Cj1="0 fF", Lj2="9 nH", Cj2="0 fF")
# example: update multiple settings
eig_qb.sim.setup_update(max_delta_f=0.4, min_freq_ghz=1.1)

eig_qb.sim.setup

{'name': 'Setup',
 'reuse_selected_design': True,
 'reuse_setup': True,
 'min_freq_ghz': 1.1,
 'n_modes': 3,
 'max_delta_f': 0.4,
 'max_passes': 6,
 'min_passes': 1,
 'min_converged': 1,
 'pct_refinement': 30,
 'basis_order': 1,
 'vars': {'Lj1': '13 nH', 'Cj1': '0 fF', 'Lj2': '9 nH', 'Cj2': '0 fF'}}

#### Execute simulation and verify convergence and EM field
Analyze a single qubit with shorted terminations. Then observe the frequency convergence plot. If not converging, you might want to increase the min_passes value to force the renderer to increase accuracy.

You can use the method `run()` instead of `sim.run()` in the following cell if you want to run both eigenmode and epr analysis in a single step. If so, make sure to also tweak the setup for the epr analysis. The input parameters are otherwise the same for the two methods.

In [22]:
eig_qb.sim.run(
    name="Qbit", components=["Q1","Q2","coupler","port_1","port_2","line_1","line_2"]#, open_terminations=[], box_plus_buffer=False
)
eig_qb.sim.plot_convergences()

INFO 08:36AM [connect_project]: Connecting to Ansys Desktop API...
INFO 08:36AM [load_ansys_project]: 	Opened Ansys App
INFO 08:36AM [load_ansys_project]: 	Opened Ansys Desktop v15.0.3
INFO 08:36AM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/AsusIran/Desktop/
	Project:   Project1
INFO 08:36AM [connect_design]: 	Opened active design
	Design:    Qbit_hfss [Solution type: Eigenmode]
INFO 08:36AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 08:36AM [connect]: 	Connected to project "Project1" and design "Qbit_hfss" 😀 

INFO 08:36AM [connect_design]: 	Opened active design
	Design:    Qbit_hfss [Solution type: Eigenmode]
INFO 08:37AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 08:37AM [analyze]: Analyzing setup Setup
08:44AM 31s INFO [get_f_convergence]: Saved convergences to C:\Users\AsusIran\Downloads\hfss_eig_f_convergence.csv


The last variables you pass to the `run()` or `sim.run()` methods, will be stored in the `sim.setup` dictionary under the key `run`. You can recall the information passed by either accessing the dictionary directly, or by using the print handle below.

In [69]:
# eig_qb.setup.run    <- direct access
eig_qb.sim.print_run_args()

This analysis object run with the following kwargs:
{'name': 'Qbit', 'components': ['Q1', 'Q2', 'coupler', 'port_1', 'port_2', 'line_1', 'line_2'], 'open_terminations': None, 'port_list': None, 'jj_to_port': None, 'ignored_jjs': None, 'box_plus_buffer': True}



In [ ]:
eig_qb.sim.convergence_f

In [ ]:
eig_qb.sim.setup.min_freq_ghz = 4
eig_qb.sim.run()
eig_qb.sim.convergence_f

INFO 08:46AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 08:46AM [analyze]: Analyzing setup Setup


In [96]:
if "jj" in eig_qb.setup.junctions:
    del eig_qb.setup.junctions["jj"]

In [97]:
eig_qb.setup.junctions.jj1 = Dict(
    rect="JJ_rect_Lj_Q1_rect_jj",
    line="JJ_Lj_Q1_rect_jj_",
    Lj_variable="Lj1",
    Cj_variable="Cj1",
)


In [98]:
eig_qb.setup.junctions.jj2 = Dict(
    rect="JJ_rect_Lj_Q2_rect_jj",
    line="JJ_Lj_Q2_rect_jj_",
    Lj_variable="Lj2",
    Cj_variable="Cj2",
)

In [99]:
eig_qb.setup.sweep_variable = "Lj1"

print(eig_qb.setup)

{'junctions': {'jj1': {'rect': 'JJ_rect_Lj_Q1_rect_jj', 'line': 'JJ_Lj_Q1_rect_jj_', 'Lj_variable': 'Lj1', 'Cj_variable': 'Cj1'}, 'jj2': {'rect': 'JJ_rect_Lj_Q2_rect_jj', 'line': 'JJ_Lj_Q2_rect_jj_', 'Lj_variable': 'Lj2', 'Cj_variable': 'Cj2'}}, 'dissipatives': {'dielectrics_bulk': ['main']}, 'cos_trunc': 8, 'fock_trunc': 7, 'sweep_variable': 'Lj1'}


Verify that the Electro(magnetic) fields look realistic.

In [100]:
eig_qb.run_epr()
eig_qb.sim.close()

Design "Qbit_hfss" info:
	# eigenmodes    3
	# variations    1
Design "Qbit_hfss" info:
	# eigenmodes    3
	# variations    1

        energy_elec_all       = 1.42020781852995e-23
        energy_elec_substrate = 1.29197172926769e-23
        EPR of substrate = 91.0%

        energy_mag    = 1.40091152564885e-23
        energy_mag % of energy_elec_all  = 98.6%
        

Variation 0  [1/1]


com_error: (-2147352567, 'Exception occurred.', (0, None, None, None, 0, -2147024381), None)

In [77]:
print(eig_qb.sim.renderer.pinfo.design.parent.parent._desktop.GetVersion())

15.0.3
